In [6]:
# ── Chargement des 4 tables nettoyées ───────────────────────────

import pandas as pd
import numpy as np

usagers    = pd.read_csv('Dataset/usagers_2021_2024_clean.csv',   sep=';', dtype={'Num_Acc': str})
caract     = pd.read_csv('Dataset/caract_2021_2024_clean.csv',    sep=';', dtype={'Num_Acc': str})
lieux      = pd.read_csv('Dataset/lieux_2021_2024_clean.csv',     sep=';', dtype={'Num_Acc': str})
vehicules  = pd.read_csv('Dataset/vehicules_2021_2024_clean.csv', sep=';', dtype={'Num_Acc': str})

print(f"usagers   : {usagers.shape}")
print(f"caract    : {caract.shape}")
print(f"lieux     : {lieux.shape}")
print(f"vehicules : {vehicules.shape}")

usagers   : (506886, 15)
caract    : (221044, 12)
lieux     : (252928, 10)
vehicules : (378071, 8)


In [7]:
# ── Vérification des clés de jointure ───────────────────────────

print("--- Types de Num_Acc ---")
for name, df in [("usagers", usagers), ("caract", caract),
                 ("lieux", lieux), ("vehicules", vehicules)]:
    print(f"  {name:<12} : {df['Num_Acc'].dtype} | NaN : {df['Num_Acc'].isna().sum()}")

print("\n--- Nb accidents uniques par table ---")
for name, df in [("usagers", usagers), ("caract", caract),
                 ("lieux", lieux), ("vehicules", vehicules)]:
    print(f"  {name:<12} : {df['Num_Acc'].nunique():,} accidents uniques")

--- Types de Num_Acc ---
  usagers      : object | NaN : 0
  caract       : object | NaN : 0
  lieux        : object | NaN : 0
  vehicules    : object | NaN : 0

--- Nb accidents uniques par table ---
  usagers      : 221,044 accidents uniques
  caract       : 221,044 accidents uniques
  lieux        : 221,044 accidents uniques
  vehicules    : 221,044 accidents uniques


In [8]:
# ── Jointure des 4 tables ───────────────────────────────────────

df = usagers.merge(caract, on='Num_Acc', how='left', suffixes=('', '_caract'))
print(f"Après merge caract    : {df.shape}")

# Dédoublonner lieux AVANT la jointure
lieux_dedup = lieux.drop_duplicates(subset=['Num_Acc'], keep='first')
print(f"lieux avant dédup : {lieux.shape[0]:,}")
print(f"lieux après dédup : {lieux_dedup.shape[0]:,}")

df = df.merge(lieux_dedup, on='Num_Acc', how='left', suffixes=('', '_lieux'))

df = df.merge(vehicules, on=['Num_Acc', 'id_vehicule'], how='left', suffixes=('', '_vehicules'))
print(f"Après merge vehicules : {df.shape}")

print(f"\n✅ Table finale : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")

Après merge caract    : (506886, 26)
lieux avant dédup : 252,928
lieux après dédup : 221,044
Après merge vehicules : (506886, 41)

✅ Table finale : 506,886 lignes x 41 colonnes


In [9]:
# ── Nettoyage post-jointure ─────────────────────────────────────

cols_to_drop = [col for col in df.columns if col.endswith('_caract')
                                           or col.endswith('_lieux')
                                           or col.endswith('_vehicules')]
df = df.drop(columns=cols_to_drop)

print(f"✅ Colonnes dupliquées supprimées")
print(f"Shape final : {df.shape}")
print(f"Colonnes : {list(df.columns)}")

✅ Colonnes dupliquées supprimées
Shape final : (506886, 37)
Colonnes : ['Num_Acc', 'id_usager', 'id_vehicule', 'num_veh', 'catu', 'grav', 'sexe', 'an_nais', 'trajet', 'secu1', 'secu2', 'secu3', 'locp', 'actp', 'annee', 'lum', 'departement', 'commune', 'agglomeration', 'intersection', 'meteo', 'type_collision', 'lat', 'long', 'date', 'catr', 'circ', 'nbv', 'prof', 'surf', 'infra', 'situ', 'vma', 'categorie_vehicule', 'obstacle_fixe', 'obstacle_mobile', 'motorisation']


In [10]:
# ── Vérification finale ─────────────────────────────────────────

print(f"Shape final : {df.shape}")

print("\n--- NaN par colonne ---")
nan_df = pd.DataFrame({
    'NaN count': df.isna().sum(),
    'NaN %': (df.isna().mean() * 100).round(1)
})
print(nan_df[nan_df['NaN count'] > 0])

print(f"\n--- Lignes perdues vs usagers ---")
print(f"Usagers avant jointure : {len(usagers):,}")
print(f"Table finale           : {len(df):,}")
print(f"Différence             : {len(usagers) - len(df):,}")

Shape final : (506886, 37)

--- NaN par colonne ---
                    NaN count  NaN %
grav                      419    0.1
sexe                    10632    2.1
an_nais                 30811    6.1
trajet                  11269    2.2
secu1                    9865    1.9
secu2                  223921   44.2
secu3                  489711   96.6
locp                   240813   47.5
lum                         7    0.0
intersection               28    0.0
meteo                      24    0.0
type_collision            207    0.0
circ                    30484    6.0
nbv                      5289    1.0
prof                      179    0.0
surf                      161    0.0
infra                    4983    1.0
situ                      160    0.0
vma                     11606    2.3
categorie_vehicule         42    0.0
obstacle_fixe             262    0.1
obstacle_mobile           225    0.0
motorisation              949    0.2

--- Lignes perdues vs usagers ---
Usagers avant jointure : 

In [11]:
# ── CELLULE 6 : Export de la table finale ───────────────────────────────────

df.to_csv('Dataset/dataset_complet_clean.csv', index=False, sep=';', encoding='utf-8')
print(f"✅ Exporté — {len(df):,} lignes x {df.shape[1]} colonnes")

✅ Exporté — 506,886 lignes x 37 colonnes
